# Predict two hazard labels, then check the official evidence

The prediction target is RECALL-MM’s hazard label field, `hazard_classification`. Inputs are product category, product name, cleaned visual product description and visible structure cues. Recall descriptions and official CPSC hazard statements are consulted afterward to understand the failure mechanism. Each notice can have multiple reference labels. Every model returns two labels.

The primary evaluation uses grouped five-fold cross-validation. The chronological test and other comparisons follow. Set `RECALL_MINILM_PATH` to load local Sentence-BERT weights.

In [1]:
import os
import re
import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer, normalize
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

SEED = 2026
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_SOURCE = os.environ.get("RECALL_MINILM_PATH", MODEL_ID)

project_dir = next((
    base for base in (Path.cwd(), Path.cwd().parent, Path.cwd() / "DM-project")
    if (base / "recall_audit/scripts/project_data.py").is_file()
), None)
if project_dir is None:
    raise FileNotFoundError("Start the notebook from the project, recall_audit, or their parent directory.")
sys.path.insert(0, str(project_dir / "recall_audit/scripts"))
from association_rules import apriori_counts, category_hazard_rules, highlight_rules
from product_features import announcement_groups, embed_variants, product_text_variants
from project_data import load_notices

records = pd.DataFrame(load_notices())
assert records["recall_number"].is_unique
dates = pd.to_datetime(records["recall_date"])
year = dates.dt.year
train = np.flatnonzero(year.between(2000, 2013))
validate = np.flatnonzero(year.between(2014, 2018))
history = np.flatnonzero(year.between(2000, 2018))
test = np.flatnonzero(year.between(2019, 2024))
assert len(np.unique(np.r_[train, validate, test])) == len(records)
category_encoder = MultiLabelBinarizer()
hazard_encoder = MultiLabelBinarizer()
categories = category_encoder.fit_transform(records["product_classification"]).astype(bool)
labels = hazard_encoder.fit_transform(records["hazard_classification"]).astype(bool)
category_names = list(category_encoder.classes_)
hazards = list(hazard_encoder.classes_)

structure_patterns = {
    "Folding mechanism or hinge": r"foldable|folding|fold[- ]lock|fold release|hinges?|hinged",
    "Wheel or caster": r"wheels|casters?|castors?",
    "Strap, harness or seat belt": r"straps?|harness(?:es)?|seat belts?",
    "Power cord or plug": r"power cords?|power cables?|electrical plugs?|power plugs?",
    "Battery compartment or cover": r"battery compartments?|battery covers?|battery doors?",
    "Drawstring or bead chain": r"drawstrings?|pull cords?|bead(?:ed)? chains?",
    "Door or lid": r"doors?|lids?|hinged covers?",
    "Vent or grille": r"air vents?|ventilation slots?|grilles?|grills?",
    "External handle": r"handles?|handgrips?|grip bars?",
}
structure_names = list(structure_patterns)
structure_re = [re.compile(rf"\b(?:{pattern})\b", re.I) for pattern in structure_patterns.values()]


fields = product_text_variants(records)
text = {column: series.map(" ".join) for column, series in fields.items()}
appearance = text["visual_product_description"]
structures = np.column_stack([
    appearance.str.contains(pattern).to_numpy() for pattern in structure_re
])
coverage = pd.DataFrame([
    {"Period": part, "Notices": len(ids),
     "Usable product name": int(text["product_name"].iloc[ids].ne("").sum()),
     "Usable visual description": int(appearance.iloc[ids].ne("").sum()),
     "Any visible structure cue": int(structures[ids].any(axis=1).sum()),
     "Multiple hazard labels": int((labels[ids].sum(axis=1) > 1).sum())}
    for part, ids in [("Fit 2000–2013", train), ("Validation 2014–2018", validate), ("Test 2019–2024", test)]
])
display(coverage)

,Period,Notices,Usable product name,Usable visual description,Any visible structure cue,Multiple hazard labels
0,Fit 2000–2013,3791,3790,3569,1287,4
1,Validation 2014–2018,1318,1317,1318,536,5
2,Test 2019–2024,1528,1528,1528,657,1


## Represent product names and appearance as vectors

In [2]:
semantic = embed_variants(fields, MODEL_SOURCE)
display(pd.Series({field: list(values.shape) for field, values in semantic.items()}, name="Notices and vector dimensions"))

product_name                  [6637, 384]
visual_product_description    [6637, 384]
Name: Notices and vector dimensions, dtype: object

## Use category frequency as a baseline

Rank hazard labels by their frequency in the training notices for that product category. Apriori is also evaluated as a separate rule-based predictor.

In [3]:
def make_features(fit_rows, query_rows):
    def design(x):
        return x[fit_rows], x[query_rows]

    category = design(categories.astype(np.float32))
    structure = design(structures.astype(np.float32))
    name, visual = [design(semantic[field]) for field in fields]

    def tfidf(column, analyzer, ngram_range):
        model = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range, min_df=2,
                                sublinear_tf=True, dtype=np.float32)
        return model.fit_transform(text[column].iloc[fit_rows]), model.transform(text[column].iloc[query_rows])

    name_word = tfidf("product_name", "word", (1, 2))
    name_char = tfidf("product_name", "char_wb", (3, 4))
    visual_word = tfidf("visual_product_description", "word", (1, 2))

    def combine(*pairs, dense=False):
        if dense:
            return tuple(np.column_stack(part) for part in zip(*pairs))
        return tuple(sparse.hstack(part, format="csr") for part in zip(*pairs))

    def pca(pair):
        model = PCA(n_components=32, svd_solver="randomized", random_state=SEED)
        return model.fit_transform(pair[0]), model.transform(pair[1])

    name_pca, visual_pca = pca(name), pca(visual)
    return {
        "TF-IDF product name": combine(category, name_word, name_char),
        "TF-IDF product name + appearance": combine(category, name_word, name_char, visual_word),
        "SBERT product name": combine(category, name, dense=True),
        "SBERT product name + appearance": combine(category, name, visual, dense=True),
        "SBERT appearance": combine(category, visual, dense=True),
        "PCA product name": combine(category, name_pca, dense=True),
        "PCA product name + appearance": combine(category, structure, name_pca, visual_pca, dense=True),
        "PCA text": combine(structure, name_pca, visual_pca, dense=True),
        "TF-IDF without product category": combine(name_word, name_char, visual_word),
    }


def prior_scores(fit_cat, fit_y, query_cat):
    num = fit_cat.astype(np.float64).T @ fit_y
    denom = fit_cat.sum(axis=0)
    global_rate = fit_y.mean(axis=0)
    rate = np.divide(num, denom[:, None], out=np.broadcast_to(global_rate, num.shape).copy(),
                     where=denom[:, None] > 0)
    counts = query_cat.sum(axis=1, keepdims=True)
    scores = np.divide(query_cat @ rate, counts,
                       out=np.broadcast_to(global_rate, (len(query_cat), len(global_rate))).copy(),
                       where=counts > 0)
    return scores, rate


def learn_rules(fit_rows, minimum_joint=5):
    train_cat, train_struct, train_y = categories[fit_rows], structures[fit_rows], labels[fit_rows]
    cat_cols = [f"C{i}" for i in range(len(category_names))]
    struct_cols = [f"S{i}" for i in range(len(structure_names))]
    hazard_cols = [f"H{i}" for i in range(len(hazards))]
    transactions = pd.DataFrame(np.column_stack([train_cat, train_struct, train_y]),
                                columns=cat_cols + struct_cols + hazard_cols)
    counts = apriori_counts(transactions, minimum_joint, max_len=3)
    details, supported = [], {}
    for c, s, h in product(range(len(cat_cols)), range(len(struct_cols)), range(len(hazard_cols))):
        antecedent = frozenset([cat_cols[c], struct_cols[s]])
        observed = counts.get(antecedent | {hazard_cols[h]}, 0)
        if not observed:
            continue
        total = counts[antecedent]
        conf = observed / total
        base = train_y[train_cat[:, c], h].mean()
        lift_in_class = conf / base
        details.append({
            "Category": category_names[c], "Structure cue": structure_names[s], "Hazard label": hazards[h],
            "Antecedent count": total, "Joint count": observed, "Confidence": conf, "Within-category lift": lift_in_class,
        })
        if total >= 10 and lift_in_class > 1:
            supported[c, s] = (total, train_y[train_cat[:, c] & train_struct[:, s]].mean(axis=0))
    return pd.DataFrame(details), supported


def rule_scores(fit_rows, query_rows, supported, shrink):
    scores, base = prior_scores(categories[fit_rows], labels[fit_rows], categories[query_rows])
    combined = scores.copy()
    for q, row in enumerate(query_rows):
        per_rule = [
            (rate - base[c]) * total / (total + shrink)
            for c in np.flatnonzero(categories[row])
            for s in np.flatnonzero(structures[row])
            if (c, s) in supported
            for total, rate in [supported[c, s]]
        ]
        if per_rule:
            combined[q] += np.mean(per_rule, axis=0)
    return np.clip(combined, 0, 1)


def top_two(scores):
    winners = np.argsort(-scores, axis=1, kind="stable")[:, :2]
    result = np.zeros(scores.shape, dtype=bool)
    result[np.arange(len(scores))[:, None], winners] = True
    return result


def score_metrics(actual, predicted):
    matched = (actual & predicted).sum(axis=1)
    return {
        "Hits": int((matched > 0).sum()),
        "Hit@2": (matched > 0).mean(),
        "micro-F1": 2 * matched.sum() / (actual.sum() + predicted.sum()),
        "macro-F1": f1_score(actual, predicted, average="macro", zero_division=0),
    }

## Compare classifiers and category routing

Fit one binary model per hazard label or score labels by neighbor votes, then rank the labels. kNN uses similar training notices as weighted votes. The Gini tree provides a single-tree comparison for XGBoost. Soft voting combines logistic regression and XGBoost scores. The optional category-routing comparison checks known and inferred category settings separately.

In [4]:
def predict_estimator(estimator, x_fit, y_fit, x_query, margin=False):
    model = OneVsRestClassifier(estimator)
    model.fit(x_fit, y_fit)
    return model.decision_function(x_query) if margin else model.predict_proba(x_query)


def learner(kind, pair, y_fit, option):
    x_fit, x_query = pair
    if kind == "Logistic regression":
        return predict_estimator(LogisticRegression(C=option, max_iter=500, solver="liblinear"),
                                 x_fit, y_fit, x_query)
    if kind == "Linear SVM":
        return predict_estimator(LinearSVC(C=option, random_state=SEED),
                                 x_fit, y_fit, x_query, margin=True)
    if kind == "Bernoulli naive Bayes":
        return predict_estimator(BernoulliNB(alpha=option), x_fit, y_fit, x_query)
    if kind == "Decision tree":
        return predict_estimator(DecisionTreeClassifier(max_depth=option, min_samples_leaf=20, random_state=SEED),
                                 x_fit, y_fit, x_query)
    if kind == "XGBoost":
        depth, rounds = option
        return predict_estimator(XGBClassifier(max_depth=depth, n_estimators=rounds, tree_method="hist",
                                                learning_rate=0.10, subsample=0.85, colsample_bytree=0.85,
                                                n_jobs=4, random_state=SEED, eval_metric="logloss"),
                                 x_fit, y_fit, x_query)
    if kind == "kNN":
        # Ask for a fixed number of neighbours and average their hazard labels.
        model = NearestNeighbors(n_neighbors=option, metric="cosine", algorithm="brute", n_jobs=4).fit(x_fit)
        distance, neighbors = model.kneighbors(x_query)
        weights = np.maximum(1 - distance, 0)
        weighted = (y_fit[neighbors] * weights[:, :, None]).sum(axis=1)
        denominator = weights.sum(axis=1, keepdims=True)
        return np.divide(weighted, denominator, out=y_fit[neighbors].mean(axis=1), where=denominator > 0)
    raise ValueError(kind)


def category_experts(pairs, fit_rows, query_rows):
    x_fit, x_query = pairs["TF-IDF without product category"]
    y_fit = labels[fit_rows]
    shared = learner("Logistic regression", (x_fit, x_query), y_fit, 1.0)
    per_class = np.broadcast_to(shared, (len(category_names), *shared.shape)).copy()
    for c in range(len(category_names)):
        use = categories[fit_rows, c]
        if use.sum() < 75:
            continue
        positive = y_fit[use].sum(axis=0)
        target = (positive >= 5) & ((use.sum() - positive) >= 5)
        if target.sum() < 2:
            continue
        per_class[c][:, target] = learner("Logistic regression", (x_fit[use], x_query), y_fit[use][:, target], 0.75)

    route_known = categories[query_rows].astype(float)
    route_known /= route_known.sum(axis=1, keepdims=True)
    routed_known = np.einsum("qc,cqh->qh", route_known, per_class)

    # Predict a class from product text only when the class is withheld.
    x_fit_tree, x_query_tree = pairs["PCA text"]
    single = categories[fit_rows].sum(axis=1) == 1
    tree = DecisionTreeClassifier(max_depth=8, min_samples_leaf=15, random_state=SEED)
    tree.fit(x_fit_tree[single], categories[fit_rows][single].argmax(axis=1))
    assigned = tree.predict(x_query_tree)
    routed_tree = per_class[assigned, np.arange(len(query_rows))]
    valid_route = categories[query_rows, assigned].mean()
    return shared, routed_known, routed_tree, valid_route


def direct_confidence_predictions(fit_rows, query_rows, book, min_antecedent, min_confidence):
    """Use eligible class-association rules first and A to fill unused slots."""
    eligible = book.loc[
        book["Antecedent count"].ge(min_antecedent) & book["Confidence"].ge(min_confidence),
        ["Category", "Structure cue", "Hazard label", "Antecedent count", "Joint count", "Confidence"],
    ]
    routes = {}
    for category, structure, hazard, denominator, joint, confidence in eligible.itertuples(index=False, name=None):
        context = (category_names.index(category), structure_names.index(structure))
        routes.setdefault(context, []).append((hazards.index(hazard), confidence, joint, denominator, context))

    a_scores, _ = prior_scores(categories[fit_rows], labels[fit_rows], categories[query_rows])
    a_order = np.argsort(-a_scores, axis=1, kind="stable")
    prediction = np.zeros((len(query_rows), len(hazards)), dtype=bool)
    traces = []

    for row_index, source in enumerate(query_rows):
        matching = [
            rule
            for c in np.flatnonzero(categories[source])
            for s in np.flatnonzero(structures[source])
            for rule in routes.get((c, s), ())
        ]
        matching.sort(key=lambda rule: (-rule[1], -rule[2], rule[0], rule[4]))
        picked = {}
        for rule in matching:
            picked.setdefault(rule[0], rule)
            if len(picked) == 2:
                break
        traces.append(picked.copy())
        for hazard_index in a_order[row_index]:
            if len(picked) == 2:
                break
            picked.setdefault(hazard_index, None)
        prediction[row_index, list(picked)] = True
    return prediction, traces, len(eligible)

## Primary evaluation with grouped five-fold cross-validation

Keep notices with the same normalized official product name and notices with confirmed links in the same group. Each notice is held out exactly once. An inner split selects model settings, including the kNN neighbor count from 7, 21 and 51. The Gini tree has fixed depth 5 and shares its inputs with XGBoost. Fit the TF-IDF vocabulary, PCA and models using their respective training partitions.

In [5]:
from sklearn.model_selection import StratifiedGroupKFold

groups = announcement_groups(records).to_numpy()
label_counts = labels.sum(axis=0)
assert labels.any(axis=1).all()
stratum = np.where(labels, label_counts, np.inf).argmin(axis=1)
all_rows = np.arange(len(records))

rule_settings = list(product((10, 20, 30), (0.15, 0.25, 0.35)))
linear_settings = (0.5, 2.0)
boost_settings = ((3, 100), (5, 100))
neighbor_settings = (7, 21, 51)
vote_weights = (0.25, 0.5, 0.75)


def grouped_splits(rows, n_splits):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for fitted, withheld in splitter.split(rows, stratum[rows], groups[rows]):
        fit_rows, query_rows = rows[fitted], rows[withheld]
        assert not set(groups[fit_rows]).intersection(groups[query_rows])
        yield fit_rows, query_rows


def cv_fit_options(fit_rows, query_rows, rules, linear, boost, neighbors):
    pairs = make_features(fit_rows, query_rows)
    fit_y = labels[fit_rows]
    book, _ = learn_rules(fit_rows)
    rule_predictions = {
        setting: direct_confidence_predictions(fit_rows, query_rows, book, *setting)[0]
        for setting in rules
    }
    linear_scores = {
        c: learner("Logistic regression", pairs["TF-IDF product name + appearance"], fit_y, c)
        for c in linear
    }
    boost_scores = {
        setting: learner("XGBoost", pairs["PCA product name + appearance"], fit_y, setting)
        for setting in boost
    }
    neighbor_scores = {
        count: learner("kNN", pairs["SBERT product name + appearance"], fit_y, count)
        for count in neighbors
    }
    return pairs, rule_predictions, linear_scores, boost_scores, neighbor_scores


def choose_by_validation(predictions, truth):
    def ranking(key):
        metrics = score_metrics(truth, predictions[key])
        return metrics["micro-F1"], metrics["macro-F1"]

    return max(predictions, key=ranking)


fold_metrics, fold_settings = [], []
oof = {}
seen = np.zeros(len(records), dtype=int)
for fold, (outer_fit, outer_test) in enumerate(grouped_splits(all_rows, 5), start=1):
    inner_fit, inner_valid = next(grouped_splits(outer_fit, 4))
    truth = labels[inner_valid]
    _, rule_candidates, lr_candidates, xgb_candidates, neighbor_candidates = cv_fit_options(
        inner_fit, inner_valid, rule_settings, linear_settings, boost_settings, neighbor_settings,
    )
    rule_choice = choose_by_validation(rule_candidates, truth)
    lr_choice = choose_by_validation({c: top_two(s) for c, s in lr_candidates.items()}, truth)
    xgb_choice = choose_by_validation({p: top_two(s) for p, s in xgb_candidates.items()}, truth)
    neighbor_choice = choose_by_validation(
        {count: top_two(scores) for count, scores in neighbor_candidates.items()}, truth
    )
    lr_valid, xgb_valid = lr_candidates[lr_choice], xgb_candidates[xgb_choice]
    vote_choice = choose_by_validation(
        {w: top_two((1-w)*lr_valid + w*xgb_valid) for w in vote_weights}, truth
    )

    pairs, rules, linear, boost, nearest = cv_fit_options(
        outer_fit, outer_test, [rule_choice], [lr_choice], [xgb_choice], [neighbor_choice],
    )
    lr_test, xgb_test = linear[lr_choice], boost[xgb_choice]
    fit_y = labels[outer_fit]
    prior, _ = prior_scores(categories[outer_fit], fit_y, categories[outer_test])
    predictions = {
        "Global frequency": top_two(np.broadcast_to(fit_y.mean(axis=0), prior.shape)),
        "A Category frequency": top_two(prior),
        "Apriori confidence": rules[rule_choice],
        "kNN": top_two(nearest[neighbor_choice]),
        "Gini decision tree": top_two(learner("Decision tree", pairs["PCA product name + appearance"], fit_y, 5)),
        "TF-IDF logistic regression": top_two(lr_test),
        "XGBoost": top_two(xgb_test),
        "Soft voting": top_two((1-vote_choice)*lr_test + vote_choice*xgb_test),
    }
    fold_settings.append({"Fold": fold, "Test notices": len(outer_test), "Rule thresholds": str(rule_choice),
                          "kNN neighbors": neighbor_choice, "Gini depth": 5,
                          "Logistic regression C": lr_choice, "XGBoost depth": xgb_choice[0],
                          "XGBoost voting weight": vote_choice})
    seen[outer_test] += 1
    for name, predicted in predictions.items():
        assert (predicted.sum(axis=1) == 2).all()
        oof.setdefault(name, np.zeros_like(labels))[outer_test] = predicted
        fold_metrics.append({"Fold": fold, "Model": name, **score_metrics(labels[outer_test], predicted)})

assert (seen == 1).all()
fold_table = pd.DataFrame(fold_metrics)
summary_cv = pd.DataFrame([
    {"Model": name, **score_metrics(labels, predicted)} for name, predicted in oof.items()
]).set_index("Model")
summary_cv["Mean fold micro-F1"] = fold_table.groupby("Model")["micro-F1"].mean()
summary_cv["Across-fold SD"] = fold_table.groupby("Model")["micro-F1"].std()
display(summary_cv.drop(index="Apriori confidence").round(4))
display(pd.DataFrame(fold_settings))
display(fold_table.loc[fold_table["Model"].ne("Apriori confidence")].pivot(
    index="Model", columns="Fold", values="micro-F1"
).round(4))

,Hits,Hit@2,micro-F1,macro-F1,Mean fold micro-F1,Across-fold SD
Model,,,,,,
Global frequency,2687,0.4049,0.2698,0.0608,0.2698,0.0004
A Category frequency,4526,0.6819,0.4546,0.3170,0.4546,0.0037
kNN,5337,0.8041,0.5361,0.4399,0.5361,0.0065
Gini decision tree,4711,0.7098,0.4731,0.3684,0.4731,0.0109
TF-IDF logistic regression,5415,0.8159,0.5438,0.4470,0.5438,0.0058
XGBoost,5317,0.8011,0.5340,0.4381,0.5340,0.0021
Soft voting,5450,0.8212,0.5474,0.4519,0.5474,0.0026


,Fold,Test notices,Rule thresholds,kNN neighbors,Gini depth,Logistic regression C,XGBoost depth,XGBoost voting weight
0,1,1328,"(10, 0.25)",21,5,2.0,3,0.50
1,2,1328,"(10, 0.15)",21,5,2.0,3,0.25
2,3,1327,"(10, 0.15)",21,5,2.0,5,0.25
3,4,1327,"(10, 0.15)",21,5,2.0,3,0.50
4,5,1327,"(10, 0.25)",21,5,2.0,5,0.75


Fold,1,2,3,4,5
Model,,,,,
A Category frequency,0.4545,0.4515,0.4505,0.4592,0.4573
Gini decision tree,0.4760,0.4660,0.4626,0.4702,0.4905
Global frequency,0.2704,0.2699,0.2697,0.2693,0.2696
Soft voting,0.5458,0.5518,0.5455,0.5461,0.5477
TF-IDF logistic regression,0.5398,0.5523,0.5374,0.5461,0.5437
XGBoost,0.5367,0.5317,0.5349,0.5320,0.5346
kNN,0.5408,0.5413,0.5274,0.5310,0.5402


Hit@2 counts a notice when either of its two predicted labels matches a reference label. Micro-F1 also counts extra predictions and missed labels.

kNN selected 21 neighbors in all five rounds and hit 5,337 of 6,637 notices. The Gini tree hit 4,711, XGBoost hit 5,317, logistic regression hit 5,415 and soft voting hit 5,450. kNN and XGBoost use different input representations. The Gini tree and XGBoost share a representation.

## Add product information one field at a time

Keep logistic regression, C=1, the folds and the two-label budget fixed. Start with product category, then add product name, visual product description and visible structure cues in turn.

In [6]:
ablation_oof = {}
for fitted, held in grouped_splits(all_rows, 5):
    pairs = make_features(fitted, held)
    text_pair = pairs["TF-IDF product name + appearance"]
    with_structures = tuple(
        sparse.hstack((values, structures[rows]), format="csr")
        for values, rows in zip(text_pair, (fitted, held), strict=True)
    )
    inputs = {
        "Logistic regression: category": (categories[fitted], categories[held]),
        "Logistic regression: add product name": pairs["TF-IDF product name"],
        "Logistic regression: add visual description": text_pair,
        "Logistic regression: add structure cues": with_structures,
    }
    estimates = {
        name: learner("Logistic regression", pair, labels[fitted], 1.0) for name, pair in inputs.items()
    }
    for name, scores in estimates.items():
        ablation_oof.setdefault(name, np.zeros_like(labels))[held] = top_two(scores)

input_effect = pd.DataFrame([
    {"Inputs": name, **score_metrics(labels, predictions)}
    for name, predictions in ablation_oof.items()
])
display(input_effect.round(4))

,Inputs,Hits,Hit@2,micro-F1,macro-F1
0,Logistic regression: category,4526,0.6819,0.4546,0.3170
1,Logistic regression: add product name,5300,0.7986,0.5323,0.4350
2,Logistic regression: add visual description,5390,0.8121,0.5413,0.4429
3,Logistic regression: add structure cues,5393,0.8126,0.5416,0.4436


Micro-F1 rises from 0.4546 with category alone to 0.5323 with name, 0.5413 with visual text and 0.5416 with structure cues. The structure cues add three hits and can also help retrieve notices about a particular component.

## Check patterns on notices held out from rule discovery

In each round, mine rules on four folds and compare the true RECALL-MM hazard labels in the held-out fold. Report pairs selected in at least three training rounds. The soft-vote predictions show whether the two-label lists reflect the same pattern; held-out labels are the reference for checking recurrence.

In [7]:
category_table = pd.DataFrame(categories, columns=category_names)
hazard_table = pd.DataFrame(labels, columns=hazards)
rule_checks = []
for fitted, held in grouped_splits(all_rows, 5):
    fitted_rules = highlight_rules(category_hazard_rules(
        category_table.iloc[fitted], hazard_table.iloc[fitted],
    ))
    for category, hazard in fitted_rules[["Product category", "Hazard label"]].itertuples(index=False, name=None):
        group = categories[held, category_names.index(category)]
        if not group.any():
            continue
        column = hazards.index(hazard)
        observed = labels[held, column]
        suggested = oof["Soft voting"][held, column]
        rule_checks.append({
            "Product category": category, "Hazard label": hazard, "Category notices": int(group.sum()),
            "Holdout notices": len(held), "Category label count": int(observed[group].sum()),
            "Holdout label count": int(observed.sum()), "Category prediction count": int(suggested[group].sum()),
            "Holdout prediction count": int(suggested.sum()),
            "Label share higher in category": observed[group].mean() > observed.mean(),
            "Prediction share higher in category": suggested[group].mean() > suggested.mean(),
        })

checks = pd.DataFrame(rule_checks)
rule_totals = []
for (category, hazard), records_for_rule in checks.groupby(["Product category", "Hazard label"]):
    if len(records_for_rule) < 3:
        continue
    totals = records_for_rule.sum(numeric_only=True)
    rule_totals.append({
        "Product category": category, "Hazard label": hazard, "Folds selecting rule": len(records_for_rule),
        "Folds with higher label share": int(totals["Label share higher in category"]),
        "Category holdout labels": f'{totals["Category label count"]}/{totals["Category notices"]}',
        "Label share: category / overall": f'{totals["Category label count"]/totals["Category notices"]:.1%} / {totals["Holdout label count"]/totals["Holdout notices"]:.1%}',
        "Folds with higher prediction share": int(totals["Prediction share higher in category"]),
        "Prediction share: category / overall": f'{totals["Category prediction count"]/totals["Category notices"]:.1%} / {totals["Holdout prediction count"]/totals["Holdout notices"]:.1%}',
    })
rule_confirmation = pd.DataFrame(rule_totals)
display(rule_confirmation)

,Product category,Hazard label,Folds selecting rule,Folds with higher label share,Category holdout labels,Label share: category / overall,Folds with higher prediction share,Prediction share: category / overall
0,BABY_PRODUCTS,CHOKING,5,5,185/402,46.0% / 14.2%,5,91.8% / 28.0%
1,CLOTHING_ACCESSORIES,CHOKING,5,5,203/425,47.8% / 14.2%,5,88.9% / 28.0%
2,ELECTRICAL,FIRE,5,5,374/594,63.0% / 24.8%,5,99.2% / 48.8%
3,FURNITURE,FALLING,5,5,271/454,59.7% / 15.6%,5,92.1% / 27.7%
4,HOME_APPLIANCES,FIRE,5,5,621/1174,52.9% / 24.8%,5,95.3% / 48.8%
5,OTHER,POISONING,5,5,132/368,35.9% / 9.4%,5,59.5% / 20.8%
6,OUTDOOR_EQUIPMENT,FIRE,5,5,172/340,50.6% / 24.8%,5,91.8% / 48.8%
7,SPORTS_RECREATION,IMPACT,5,5,148/619,23.9% / 7.3%,5,71.4% / 16.2%
8,TOOLS_HARDWARE,CUTS,5,5,123/346,35.5% / 6.7%,5,72.5% / 12.3%
9,TOYS_CHILDREN,CHOKING,5,5,493/1168,42.2% / 14.2%,5,86.6% / 28.0%


All 11 representative rules were selected in every training round, and their labels remained more common in the category than overall in each held-out fold. For FIRE, the shares are 374/594 for electrical products, 621/1,174 for home appliances and 24.8% overall.

## Trace a prediction back to a component

Use the held-out Tern folding bicycle notice as an example. In that fold’s training notices, compare transportation products named bicycle or bicycles that have usable appearance text and one source row. Count the FALLING label by whether the visual description mentions folding or a hinge. Read the official notice to identify the actual failure mechanism.

In [8]:
case_index = int(records.index[records["recall_number"].eq("13-231")].item())
reference, _ = next((fitted, held) for fitted, held in grouped_splits(all_rows, 5) if case_index in held)
assert groups[case_index] not in groups[reference]
bicycles = text["product_name"].str.contains(r"\bbicycles?\b", case=False, regex=True)
in_transport = categories[:, category_names.index("TRANSPORTATION")]
single_source = records["source_rows"].map(len).eq(1)
usable_appearance = appearance.ne("")
comparable = reference[(bicycles & in_transport & single_source & usable_appearance).to_numpy()[reference]]
hinge = structures[:, structure_names.index("Folding mechanism or hinge")]
fall = labels[:, hazards.index("FALLING")]
display(pd.DataFrame([{
    "Holdout notices": records.at[case_index, "recall_number"],
    "Two predicted hazard labels": ", ".join(np.array(hazards)[oof["Soft voting"][case_index]]),
    "Reference hazard labels": ", ".join(records.at[case_index, "hazard_classification"]),
}]))
counts = []
for mentioned, ids in (("Mentions folding or a hinge", comparable[hinge[comparable]]),
                       ("No folding or hinge mention", comparable[~hinge[comparable]])):
    counts.append({"Reference notices": mentioned, "FALLING label count": int(fall[ids].sum()), "Notice count": len(ids),
                   "Share": float(fall[ids].mean()) if len(ids) else np.nan})
design_counts = pd.DataFrame(counts)
display(design_counts.round(3))
matching = records.loc[comparable[hinge[comparable] & fall[comparable]],
                       ["recall_number", "product_name", "official"]].copy()
matching["Failure location or mechanism in official notice"] = matching["official"].map(lambda item: item["hazard_description"])
matching["Official notice URL"] = matching["official"].map(lambda item: item["url"])
with pd.option_context("display.max_colwidth", 200, "display.max_rows", 14):
    display(matching.drop(columns="official"))

,Holdout notices,Two predicted hazard labels,Reference hazard labels
0,13-231,"CRASH, FALLING",FALLING


,Reference notices,FALLING label count,Notice count,Share
0,Mentions folding or a hinge,7,7,1.000
1,No folding or hinge mention,102,151,0.675


,recall_number,product_name,Failure location or mechanism in official notice,Official notice URL
2463,09-214,2008 model year folding bicycles,"The hinge on the handlepost can crack, posing a fall hazard.",https://www.cpsc.gov/Recalls/2009/Dahon-and-REI-Novara-Folding-Bicycles-Recalled-Due-to-Fall-Hazard
3494,13-038,Folding Bicycles,"The tikit bike's handlebar stem can break and cause the rider to lose control, posing a fall hazard to the consumer.",https://www.cpsc.gov/Recalls/2013/Bike-Friday-Recalls-Tikit-Folding-Bicycles-Due-to-Fall-Hazard
3813,14-096,folding bicycles,"The bicycle's frame can break during use, posing a fall hazard to the rider.",https://www.cpsc.gov/Recalls/2014/West-Marine-Recalls-Folding-Bicycles
4336,16-068,folding bicycles,"The frame on the folding bicycles can break, posing a fall hazard.",https://www.cpsc.gov/Recalls/2016/Origin8-Recalls-Folding-Bicycles
4591,17-708,Runner bicycles with detachable pedals,"The pedals can detach from the bionic runner bicycle, posing a fall hazard to riders.&nbsp;",https://www.cpsc.gov/Recalls/2017/Running-Trainer-Recalls-Bicycles
4863,18-074,folding bicycles,"The bicycle’s bottom bracket can break during use, posing a fall hazard.",https://www.cpsc.gov/Recalls/2018/Brompton-Bicycle-Recalls-Bicycles-Due-To-Fall-Hazard
4907,18-108,Folding Bicycles,"The bike’s frame can break during use causing the bicycle to collapse, posing a fall hazard to consumers.",https://www.cpsc.gov/Recalls/2018/Allen-Sports-Recalls-Folding-Bicycles-Due-to-Fall-Hazard


The two predicted labels are CRASH and FALLING. The reference label is FALLING. In the training reference set, 7/7 notices that mention folding or a hinge carry FALLING, versus 102/151 without a mention. The seven cases are a retrieval clue. The Tern notice identifies possible cracking at the top-tube hinge. A design review of a similar mechanism can check whether that connection cracks under repeated loading. [Tern 13-231](https://www.cpsc.gov/Recalls/2013/Stile-Products-Recalls-Tern-Folding-Bicycles)

## Supplementary chronological evaluation

Fit on 2000–2013, select settings on 2014–2018, then refit through 2018 and test on 2019–2024.

## Select settings on the validation period

In [9]:
def config(kind, feature, option):
    return (kind, feature, option)


CONFIGS = (
    [config("Bernoulli naive Bayes", "TF-IDF product name + appearance", alpha) for alpha in (0.5, 1.0)]
    + [config(kind, feat, c) for kind in ("Logistic regression", "Linear SVM")
       for feat in ("TF-IDF product name", "TF-IDF product name + appearance", "SBERT product name", "SBERT product name + appearance", "SBERT appearance")
       for c in (0.5, 2.0)]
    + [config("Decision tree", "PCA product name + appearance", depth) for depth in (5, 9)]
    + [config("kNN", feat, k) for feat in ("TF-IDF product name + appearance", "SBERT product name + appearance") for k in (7, 21, 51)]
    + [config("XGBoost", feat, setting) for feat in ("PCA product name", "PCA product name + appearance")
       for setting in ((3, 100), (5, 100))]
)


def evaluation_row(name, feature, parameter, scores, truth):
    return {"Model": name, "Features": feature, "Parameters": str(parameter), **score_metrics(truth, top_two(scores))}


def select_one(rows):
    table = pd.DataFrame(rows)
    return int(table.sort_values(["micro-F1", "macro-F1"], ascending=False, kind="stable").index[0])


def fit_and_score(fit_rows, query_rows, model_config):
    pairs = make_features(fit_rows, query_rows)
    outputs = {}
    y_fit = labels[fit_rows]
    outputs["Global frequency"] = np.broadcast_to(y_fit.mean(axis=0), labels[query_rows].shape)
    outputs["A Category prior"], _ = prior_scores(categories[fit_rows], y_fit, categories[query_rows])
    rule_book, supported = learn_rules(fit_rows)
    for shrink in (10, 30):
        outputs[("Rules", shrink)] = rule_scores(fit_rows, query_rows, supported, shrink)
    for run in model_config:
        kind, feat, option = run
        outputs[run] = learner(kind, pairs[feat], y_fit, option)
    shared, known, routed, routing_hit = category_experts(pairs, fit_rows, query_rows)
    for weight in (0.5, 0.85):
        outputs[("Known-category experts", weight)] = shared * (1 - weight) + known * weight
        outputs[("Gini category routing", weight)] = shared * (1 - weight) + routed * weight
    outputs["Global model without category"] = shared
    return outputs, rule_book, routing_hit


VALID = {part: labels[ids] for part, ids in [("val", validate), ("test", test)]}
val_outputs, rule_book, validation_routing = fit_and_score(train, validate, CONFIGS)
val_rows = []
val_key = []
for key, scores in val_outputs.items():
    if isinstance(key, str):
        row = evaluation_row(key, "", "", scores, VALID["val"])
    elif len(key) == 2:
        row = evaluation_row(key[0], "Category + structure cues" if key[0] == "Rules" else "Product name + appearance with category routing",
                             key[1], scores, VALID["val"])
    else:
        row = evaluation_row(*key, scores, VALID["val"])
    val_rows.append(row)
    val_key.append(key)

# Compare soft-voting options on validation only, using each family's chosen configuration.
by_family = {}
for family in sorted(set(row["Model"] for row in val_rows)):
    family_positions = [i for i, row in enumerate(val_rows) if row["Model"] == family]
    by_family[family] = val_key[family_positions[select_one([val_rows[i] for i in family_positions])]]

linear_key, boost_key = by_family["Logistic regression"], by_family["XGBoost"]
for weight in (0.25, 0.5, 0.75):
    key = ("Soft voting", weight)
    scores = (1 - weight) * val_outputs[linear_key] + weight * val_outputs[boost_key]
    val_outputs[key] = scores
    val_rows.append(evaluation_row(key[0], "Best validation logistic regression + XGBoost", weight, scores, VALID["val"]))
    val_key.append(key)
vote_positions = [i for i, key in enumerate(val_key) if isinstance(key, tuple) and key[0] == "Soft voting"]
by_family["Soft voting"] = val_key[vote_positions[select_one([val_rows[i] for i in vote_positions])]]
validation = pd.DataFrame(val_rows)
display(validation.sort_values(["micro-F1", "macro-F1"], ascending=False, kind="stable").round(4))
display(rule_book.sort_values(["Joint count", "Within-category lift"], ascending=False).head(12).round(3))

,Model,Features,Parameters,Hits,Hit@2,micro-F1,macro-F1
43,Soft voting,Best validation logistic regression + XGBoost,0.25,1027,0.7792,0.5198,0.4135
44,Soft voting,Best validation logistic regression + XGBoost,0.5,1015,0.7701,0.5138,0.4109
9,Logistic regression,TF-IDF product name + appearance,2.0,1011,0.7671,0.5117,0.4137
38,Known-category experts,Product name + appearance with category routing,0.5,1008,0.7648,0.5102,0.4070
45,Soft voting,Best validation logistic regression + XGBoost,0.75,1006,0.7633,0.5092,0.4082
13,Logistic regression,SBERT product name + appearance,2.0,1005,0.7625,0.5087,0.4090
12,Logistic regression,SBERT product name + appearance,0.5,1001,0.7595,0.5067,0.4024
40,Known-category experts,Product name + appearance with category routing,0.85,997,0.7564,0.5047,0.4068
11,Logistic regression,SBERT product name,2.0,996,0.7557,0.5042,0.4043
32,kNN,SBERT product name + appearance,21,996,0.7557,0.5042,0.3991


,Category,Structure cue,Hazard label,Antecedent count,Joint count,Confidence,Within-category lift
10,CLOTHING_ACCESSORIES,Drawstring or bead chain,CHOKING,95,80,0.842,1.509
30,HOME_APPLIANCES,External handle,FIRE,136,61,0.449,0.777
25,HOME_APPLIANCES,Door or lid,FIRE,110,51,0.464,0.803
28,HOME_APPLIANCES,External handle,BURN,136,43,0.316,2.159
27,HOME_APPLIANCES,Vent or grille,FIRE,54,43,0.796,1.379
66,TOYS_CHILDREN,External handle,CHOKING,100,41,0.410,0.962
53,TOOLS_HARDWARE,External handle,CUTS,95,40,0.421,1.117
58,TOYS_CHILDREN,Wheel or caster,CHOKING,85,33,0.388,0.910
44,SPORTS_RECREATION,"Strap, harness or seat belt",FALLING,62,30,0.484,1.199
70,TOYS_CHILDREN,External handle,POISONING,100,30,0.300,1.032


## Evaluate on later notices

In [10]:
# The design task assumes category membership is available. Keep the inferred route as a separate scenario.
primary = [name for name in by_family if name not in ("Gini category routing", "Global model without category")]
selected = [by_family[name] for name in primary]
where_selected = [val_key.index(key) for key in selected]
winner = selected[select_one([val_rows[i] for i in where_selected])]
display(pd.DataFrame([val_rows[val_key.index(winner)]]).round(4).rename(index={0: "Selected on validation"}))

# Retrain the already-chosen configurations on all pre-2019 history.
refit_configs = [key for key in selected if isinstance(key, tuple) and len(key) == 3]
test_outputs, full_rules, test_routing = fit_and_score(history, test, refit_configs)
vote_group, weight = by_family["Soft voting"]
test_outputs[by_family["Soft voting"]] = (1 - weight) * test_outputs[linear_key] + weight * test_outputs[boost_key]

chosen = [*selected, by_family["Gini category routing"], "Global model without category"]
test_rows = []
for key in chosen:
    row = val_rows[val_key.index(key)].copy()
    for metric in ["Hits", "Hit@2", "micro-F1", "macro-F1"]:
        row.pop(metric)
    row["Selected on validation"] = "Yes" if key == winner else ""
    test_rows.append({**row, **score_metrics(VALID["test"], top_two(test_outputs[key]))})
test_table = pd.DataFrame(test_rows).sort_values("micro-F1", ascending=False, kind="stable")
display(test_table[["Model", "Features", "Parameters", "Selected on validation", "Hits", "Hit@2", "micro-F1", "macro-F1"]].round(4))
display(pd.Series({"Validation category routing hit rate": validation_routing,
                   "Test category routing hit rate": test_routing}, name="Text-only Gini category prediction").to_frame())

,Model,Features,Parameters,Hits,Hit@2,micro-F1,macro-F1
Selected on validation,Soft voting,Best validation logistic regression + XGBoost,0.25,1027,0.7792,0.5198,0.4135


,Model,Features,Parameters,Selected on validation,Hits,Hit@2,micro-F1,macro-F1
10,Soft voting,Best validation logistic regression + XGBoost,0.25,Yes,1222,0.7997,0.5330,0.4461
6,Logistic regression,TF-IDF product name + appearance,2.0,,1212,0.7932,0.5287,0.4358
4,Known-category experts,Product name + appearance with category routing,0.5,,1202,0.7866,0.5243,0.4312
8,XGBoost,PCA product name + appearance,"(5, 100)",,1196,0.7827,0.5217,0.4424
12,Global model without category,,,,1194,0.7814,0.5208,0.4335
9,kNN,SBERT product name + appearance,21,,1190,0.7788,0.5191,0.4256
1,Bernoulli naive Bayes,TF-IDF product name + appearance,0.5,,1181,0.7729,0.5152,0.4097
5,Linear SVM,SBERT product name + appearance,0.5,,1172,0.7670,0.5112,0.4314
11,Gini category routing,Product name + appearance with category routing,0.5,,1165,0.7624,0.5082,0.4152
2,Decision tree,PCA product name + appearance,5,,1037,0.6787,0.4523,0.3530


,Text-only Gini category prediction
Validation category routing hit rate,0.631259
Test category routing hit rate,0.622382


## Examine gains and misses by hazard label

Compare correct suggestions by hazard label, paired bootstrap differences and case examples.

In [11]:
predicted = top_two(test_outputs[winner])
baseline = top_two(test_outputs["A Category prior"])
actual = VALID["test"]
results = pd.DataFrame({
    "Hazard label": hazards,
    "Reference count": actual.sum(axis=0),
    "A hits": (baseline & actual).sum(axis=0),
    "Selected model hits": (predicted & actual).sum(axis=0),
})
results["Change in hits"] = results["Selected model hits"] - results["A hits"]
display(results.sort_values("Reference count", ascending=False).reset_index(drop=True))

rng = np.random.default_rng(SEED)
draws = rng.integers(len(test), size=(2000, len(test)))
denominator = actual.sum(axis=1) + 2
actual_tp = (predicted & actual).sum(axis=1)
bootstrap_results = []
for reference, predictions in [
    ("A Category prior", baseline), ("Validation-selected logistic regression", top_two(test_outputs[linear_key]))
]:
    match_difference = actual_tp - (predictions & actual).sum(axis=1)
    delta = 2 * match_difference[draws].sum(axis=1) / denominator[draws].sum(axis=1)
    observed_delta = 2 * match_difference.sum() / denominator.sum()
    interval = np.quantile(delta, [0.025, 0.975])
    bootstrap_results.append({"Reference": reference, "micro-F1 difference": observed_delta,
                              "Paired bootstrap lower bound": interval[0], "Paired bootstrap upper bound": interval[1]})
interval_table = pd.DataFrame(bootstrap_results).set_index("Reference")
display(interval_table.round(4))

followup = records["official"].map(lambda item: item["title"]).iloc[test].str.contains(
    r"\b(?:expand\w*|re[- ]?announc\w*)\b", regex=True, case=False
).to_numpy()
without = ~followup
if (~without).any():
    display(pd.DataFrame([
        {"Subset": "Titles without expansion or reannouncement", "Notice count": int(without.sum()),
         "Model": name, **score_metrics(actual[without], matrix[without])}
        for name, matrix in [("A Category prior", baseline), ("Selected on validation", predicted)]
    ]).round(4))

def decode(matrix):
    return [", ".join(h for h, found in zip(hazards, row) if found) for row in matrix]
qualitative = records.iloc[test][["recall_number", "recall_date", "product_name"]].reset_index(drop=True)
qualitative["Reference hazard labels"] = decode(actual)
qualitative["A predictions"] = decode(baseline)
qualitative["Selected model predictions"] = decode(predicted)
winner_hit, a_hit = (predicted & actual).any(axis=1), (baseline & actual).any(axis=1)
display(qualitative.loc[winner_hit & ~a_hit].head(6))
display(qualitative.loc[a_hit & ~winner_hit].head(6))

,Hazard label,Reference count,A hits,Selected model hits,Change in hits
0,FIRE,352,275,337,62
1,FALLING,226,191,210,19
2,POISONING,201,151,168,17
3,CHOKING,143,133,137,4
4,CRASH,139,109,113,4
5,BURN,137,90,112,22
6,IMPACT,113,26,57,31
7,CUTS,86,26,47,21
8,OTHER,85,0,22,22
9,SAFETY_EQUIPMENT_MALFUNCTION,39,0,19,19


,micro-F1 difference,Paired bootstrap lower bound,Paired bootstrap upper bound
Reference,,,
A Category prior,0.0964,0.0833,0.1095
Validation-selected logistic regression,0.0044,-0.0013,0.0100


,Subset,Notice count,Model,Hits,Hit@2,micro-F1,macro-F1
0,Titles without expansion or reannouncement,1514,A Category prior,994,0.6565,0.4376,0.3111
1,Titles without expansion or reannouncement,1514,Selected on validation,1213,0.8012,0.5340,0.4470


,recall_number,recall_date,product_name,Reference hazard labels,A predictions,Selected model predictions
11,19-070,2019-02-21,yellow diving pressure transmitters,OTHER,"FALLING, IMPACT","OTHER, SAFETY_EQUIPMENT_MALFUNCTION"
18,19-726,2019-03-05,off-highway utility vehicles,FIRE,"CRASH, FALLING","CRASH, FIRE"
22,19-080,2019-03-13,Sauté Pan with glass lid,CUTS,"BURN, FIRE","BURN, CUTS"
24,19-082,2019-03-14,Performance heated socks,FIRE,"BURN, CHOKING","BURN, FIRE"
31,19-086,2019-03-21,Wall-mounted range hoods,FALLING,"BURN, FIRE","FALLING, FIRE"
32,19-087,2019-03-21,Gun safe,SAFETY_EQUIPMENT_MALFUNCTION,"FIRE, POISONING","IMPACT, SAFETY_EQUIPMENT_MALFUNCTION"


,recall_number,recall_date,product_name,Reference hazard labels,A predictions,Selected model predictions
12,19-071,2019-02-26,folding mini trampolines,IMPACT,"FALLING, IMPACT","CUTS, FALLING"
13,19-072,2019-02-26,Sleeper Chair Folding Foam Beds,FIRE,"FALLING, FIRE","CUTS, FALLING"
131,19-172,2019-07-24,10 inch table saw,FIRE,"CUTS, FIRE","CUTS, IMPACT"
143,19-180,2019-08-07,Ice Axes,FALLING,"FALLING, IMPACT","CUTS, IMPACT"
182,19-211,2019-09-26,Multi-Functional Utility Bars,CUTS,"CUTS, FIRE","FIRE, IMPACT"
233,20-712,2020-01-09,children's tricycles with canopy,POISONING,"CHOKING, POISONING","CHOKING, FALLING"


## Interpret the chronological test

Soft voting hits 1,222 of the 1,528 later notices versus 1,212 for the validation-selected logistic regression. The paired micro-F1 interval for the difference includes zero. The test measures label coverage on historical notices, which can guide what to inspect. Estimating incident rates requires exposure and incident data.

## Supplementary Apriori prediction comparison

Select rules using the training and validation records, then rank applicable rules by confidence. Fill unused prediction slots with category frequency. Rule confidence measures how often the hazard label co-occurs with the antecedent in the data.

In [12]:
display(summary_cv.loc[["A Category frequency", "Apriori confidence"]].round(4))

# Select thresholds using the same validation period as the earlier experiment.
confidence_fit_book, shrink_fit = learn_rules(train)
confidence_validation = []
for min_antecedent, min_confidence in product((10, 20, 30), (0.15, 0.25, 0.35)):
    prediction, traces, n_rules = direct_confidence_predictions(
        train, validate, confidence_fit_book, min_antecedent, min_confidence
    )
    confidence_validation.append({
        "Minimum antecedent count": min_antecedent, "Minimum confidence": min_confidence, "Rule count": n_rules,
        "At least one rule used": sum(bool(row) for row in traces),
        **score_metrics(labels[validate], prediction),
    })

confidence_validation = pd.DataFrame(confidence_validation).sort_values(
    ["micro-F1", "macro-F1", "Minimum antecedent count", "Minimum confidence"],
    ascending=[False, False, False, False],
)
best_confidence = confidence_validation.iloc[0]
confidence_parameters = int(best_confidence["Minimum antecedent count"]), float(best_confidence["Minimum confidence"])
display(confidence_validation.round(4).reset_index(drop=True))

# Reproduce the previous rule variant's selection on validation, for the side-by-side comparison.
shrink_validation = pd.DataFrame([
    {"Shrinkage parameter": shrink, **score_metrics(labels[validate], top_two(rule_scores(train, validate, shrink_fit, shrink)))}
    for shrink in (10, 30)
])
shrink_selection = int(shrink_validation.sort_values(
    ["micro-F1", "macro-F1"], ascending=False, kind="stable"
).iloc[0]["Shrinkage parameter"])

# Refit counts through 2018. Keep the early-selected thresholds for the 2019–2024 evaluation.
confidence_history_book, shrink_history = learn_rules(history)
confidence_test, confidence_trace, history_rule_count = direct_confidence_predictions(
    history, test, confidence_history_book, *confidence_parameters,
)
base_test, _ = prior_scores(categories[history], labels[history], categories[test])
confidence_comparators = {
    "A Category prior": top_two(base_test),
    "Apriori with category shrinkage": top_two(rule_scores(history, test, shrink_history, shrink_selection)),
    "Apriori ranked by confidence": confidence_test,
}
confidence_baseline_table = pd.DataFrame([
    {"Model": model, **score_metrics(labels[test], matrix)}
    for model, matrix in confidence_comparators.items()
])
display(confidence_baseline_table.round(4))
display(pd.Series({
    "Validation-selected minimum antecedent count": confidence_parameters[0],
    "Validation-selected minimum confidence": confidence_parameters[1],
    "Rules refitted on historical records": history_rule_count,
    "Test notices with at least one rule prediction": sum(bool(record) for record in confidence_trace),
    "Test notices with two rule predictions": sum(len(record) == 2 for record in confidence_trace),
}, name="Direct confidence ranking", dtype=object))

# Quantify the paired difference using the same bootstrap unit as the earlier report.
confidence_truth = labels[test]
denominator = confidence_truth.sum(axis=1) + 2
rng = np.random.default_rng(SEED)
bootstrap_rows = rng.integers(len(test), size=(2000, len(test)))
confidence_deltas = []
for model, other in confidence_comparators.items():
    if model == "Apriori ranked by confidence":
        continue
    difference = (confidence_test & confidence_truth).sum(axis=1) - (other & confidence_truth).sum(axis=1)
    draws = 2 * difference[bootstrap_rows].sum(axis=1) / denominator[bootstrap_rows].sum(axis=1)
    lower, upper = np.quantile(draws, [0.025, 0.975])
    confidence_deltas.append({
        "Reference": model, "micro-F1 difference": 2 * difference.sum() / denominator.sum(),
        "95% interval lower bound": lower, "95% interval upper bound": upper,
    })
confidence_intervals = pd.DataFrame(confidence_deltas)
display(confidence_intervals.round(4))

# Show which actual rules moved a hazard into the top two.
rule_examples = []
actual_hazard_names = decode(confidence_truth)
from_a = confidence_comparators["A Category prior"]
for q, picked in enumerate(confidence_trace):
    for hazard_index, (_, confidence, joint, total, (category_index, structure_index)) in picked.items():
        if from_a[q, hazard_index]:
            continue
        rule_examples.append({
            "Notice number": records.iloc[test[q]]["recall_number"],
            "Product": text["product_name"].iloc[test[q]],
            "Category": category_names[category_index], "Structure cue": structure_names[structure_index],
            "New hazard label": hazards[hazard_index], "Historical joint / antecedent count": f"{joint}/{total}",
            "Historical confidence": confidence, "Reference hazard labels": actual_hazard_names[q],
            "Hit": bool(confidence_truth[q, hazard_index]),
        })
confidence_examples = pd.DataFrame(rule_examples)
if not confidence_examples.empty:
    display(confidence_examples.groupby("Hit", sort=False).head(4).round(3).reset_index(drop=True))

,Hits,Hit@2,micro-F1,macro-F1,Mean fold micro-F1,Across-fold SD
Model,,,,,,
A Category frequency,4526,0.6819,0.4546,0.3170,0.4546,0.0037
Apriori confidence,4580,0.6901,0.4600,0.3216,0.4600,0.0035


,Minimum antecedent count,Minimum confidence,Rule count,At least one rule used,Hits,Hit@2,micro-F1,macro-F1
0,30,0.15,34,357,855,0.6487,0.4319,0.3162
1,20,0.15,40,390,855,0.6487,0.4319,0.3155
2,10,0.15,55,473,855,0.6487,0.4319,0.3154
3,30,0.35,18,357,851,0.6457,0.4299,0.3146
4,30,0.25,22,357,851,0.6457,0.4299,0.3146
5,20,0.35,23,390,851,0.6457,0.4299,0.3146
6,10,0.35,37,473,851,0.6457,0.4299,0.3145
7,20,0.25,28,390,851,0.6457,0.4299,0.3139
8,10,0.25,43,473,851,0.6457,0.4299,0.3138


,Model,Hits,Hit@2,micro-F1,macro-F1
0,A Category prior,1001,0.6551,0.4366,0.3100
1,Apriori with category shrinkage,1025,0.6708,0.4471,0.3219
2,Apriori ranked by confidence,1026,0.6715,0.4475,0.3146


Validation-selected minimum antecedent count        30
Validation-selected minimum confidence            0.15
Rules refitted on historical records                43
Test notices with at least one rule prediction     513
Test notices with two rule predictions             454
Name: Direct confidence ranking, dtype: object

,Reference,micro-F1 difference,95% interval lower bound,95% interval upper bound
0,A Category prior,0.0109,0.0061,0.0161
1,Apriori with category shrinkage,0.0004,-0.0044,0.0048


,Notice number,Product,Category,Structure cue,New hazard label,Historical joint / antecedent count,Historical confidence,Reference hazard labels,Hit
0,19-728,off-highway utility vehicles,TRANSPORTATION,Vent or grille,FIRE,16/52,0.308,FIRE,True
1,19-737,walk-behind lawn mowers,OUTDOOR_EQUIPMENT,External handle,CUTS,11/56,0.196,IMPACT,False
2,19-117,Marine coolers,OUTDOOR_EQUIPMENT,External handle,CUTS,11/56,0.196,OTHER,False
3,19-746,recreational off-highway vehicles,TRANSPORTATION,Vent or grille,FIRE,16/52,0.308,CRASH,False
4,19-748,Zero turn lawn mowers,OUTDOOR_EQUIPMENT,Wheel or caster,CUTS,13/75,0.173,FIRE,False
5,20-733,Off-road vehicles with two or four seats,TRANSPORTATION,Vent or grille,FIRE,16/52,0.308,FIRE,True
6,20-734,Off-road vehicles with two or four seats,TRANSPORTATION,Vent or grille,FIRE,16/52,0.308,FIRE,True
7,20-121,baby and toddler swing,TOYS_CHILDREN,"Strap, harness or seat belt",FALLING,10/35,0.286,FALLING,True
